# nuScenes-Night Metric Depth Evaluation

This notebook evaluates five input pipelines using **DepthAnythingV2 Metric Outdoor Small**:

- Dark baseline
- Gamma correction
- CLAHE
- MSR
- LLFormer

The **108-image low-brightness subset** is the primary evaluation set.  
The **200-image random nighttime sample** is retained as a supplementary robustness analysis.

Depth is evaluated directly in metres against sparse nuScenes LiDAR reference depth using MAE, RMSE, and AbsRel. No scale alignment is applied.


## 1. Environment Setup

Run this section first in a fresh Kaggle session. Internet access must be enabled.


In [1]:
import os
import sys
import random
import shutil
import subprocess
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image
from tqdm import tqdm
from skimage import img_as_ubyte
from collections import OrderedDict

WORK_DIR = Path("/kaggle/working")
DEPTH_REPO = WORK_DIR / "Depth-Anything-V2"
LLFORMER_REPO = WORK_DIR / "LLFormer"


def run_command(command, cwd=None):
    """Run a shell command and display its output."""
    print(f"\nRunning: {command}")
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}:\n{command}"
        )


if not DEPTH_REPO.is_dir():
    run_command(
        "git clone https://github.com/DepthAnything/Depth-Anything-V2.git",
        cwd=WORK_DIR
    )
else:
    print(f"Depth Anything V2 already exists: {DEPTH_REPO}")


if not LLFORMER_REPO.is_dir():
    run_command(
        "git clone https://github.com/TaoWangzj/LLFormer.git",
        cwd=WORK_DIR
    )
else:
    print(f"LLFormer already exists: {LLFORMER_REPO}")


warmup_dir = LLFORMER_REPO / "pytorch-gradual-warmup-lr"
if warmup_dir.is_dir():
    run_command(
        f'"{sys.executable}" setup.py install',
        cwd=warmup_dir
    )
else:
    run_command(
        f'"{sys.executable}" -m pip install pytorch-gradual-warmup-lr -q'
    )


run_command(
    f'"{sys.executable}" -m pip install '
    f'natsort yacs gdown transformers scikit-image h5py joblib -q'
)


for repo_path in [str(DEPTH_REPO), str(LLFORMER_REPO)]:
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

print("\nEnvironment setup completed.")



Running: git clone https://github.com/DepthAnything/Depth-Anything-V2.git
Cloning into 'Depth-Anything-V2'...


Running: git clone https://github.com/TaoWangzj/LLFormer.git
Cloning into 'LLFormer'...


Running: "/usr/bin/python3" setup.py install
running install
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        This deprecation is overdue, please update your project and remove deprecated
        calls to avoid build errors in the future.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
running build
run

## 2. Download LLFormer LOL Weights


In [2]:
from pathlib import Path
import subprocess
import sys

LLFORMER_CHECKPOINT_DIR = (
    Path("/kaggle/working/LLFormer/checkpoints/LOL")
)
LLFORMER_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def find_llformer_checkpoint(root_dir):
    candidates = list(
        root_dir.rglob("model_bestPSNR.pth")
    )

    if not candidates:
        return None

    candidates.sort(
        key=lambda path: (
            "models" not in path.parts,
            len(path.parts)
        )
    )
    return candidates[0]


weights_path = find_llformer_checkpoint(
    LLFORMER_CHECKPOINT_DIR
)

if weights_path is None:
    command = [
        sys.executable,
        "-m",
        "gdown",
        "--folder",
        "https://drive.google.com/drive/folders/"
        "1J7NvvPsCtT0j8Rd9ombJ6sVIC6v0Xweb",
        "-O",
        str(LLFORMER_CHECKPOINT_DIR)
    ]

    print("Downloading LLFormer LOL weights...")

    result = subprocess.run(
        command,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            "LLFormer weight download failed."
        )

    weights_path = find_llformer_checkpoint(
        LLFORMER_CHECKPOINT_DIR
    )


if weights_path is None:
    print("\nDownloaded files:")
    for path in LLFORMER_CHECKPOINT_DIR.rglob("*"):
        if path.is_file():
            print(path)

    raise FileNotFoundError(
        "model_bestPSNR.pth could not be found "
        "under the LLFormer checkpoint directory."
    )


print("LLFormer checkpoint ready:")
print(weights_path)


Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1nTLY_gChn2_Va2BfS_R_W-bdjsqsLHJB
From (redirected): https://drive.google.com/uc?id=1nTLY_gChn2_Va2BfS_R_W-bdjsqsLHJB&confirm=t&uuid=2a99cae0-d3ef-4f1c-a246-9c1fa3e0445c
To: /kaggle/working/LLFormer/checkpoints/LOL/models/model_bestPSNR.pth
  0%|          | 0.00/296M [00:00<?, ?B/s]

Retrieving folder 1AZchyGAR5vtQpIEZTdDYH9t605nk_YBR models
Processing file 1nTLY_gChn2_Va2BfS_R_W-bdjsqsLHJB model_bestPSNR.pth
Processing file 1S7FlcG-aEMUfDEsh9ZdcMnc5G0sfE4MB model_bestSSIM.pth


100%|██████████| 296M/296M [00:02<00:00, 108MB/s]  
Downloading...
From (original): https://drive.google.com/uc?id=1S7FlcG-aEMUfDEsh9ZdcMnc5G0sfE4MB
From (redirected): https://drive.google.com/uc?id=1S7FlcG-aEMUfDEsh9ZdcMnc5G0sfE4MB&confirm=t&uuid=1eb4803a-4980-400f-9bb6-6124bf39ee62
To: /kaggle/working/LLFormer/checkpoints/LOL/models/model_bestSSIM.pth
100%|██████████| 296M/296M [00:02<00:00, 126MB/s]  
Download completed


LLFormer checkpoint ready:
/kaggle/working/LLFormer/checkpoints/LOL/models/model_bestPSNR.pth


## 3. Dataset Paths and Sample Definitions

The 108-image dataset is used for the primary low-light analysis.  
The random 200-image sample is used only for supplementary robustness analysis.


In [3]:
# Primary low-brightness subset
COLOR_DIR_DARK = Path(
    "/kaggle/input/datasets/mavislei/"
    "nuscenes-dark-108/nuscenes_night_dark_subset_108/color"
)
GT_DIR_DARK = Path(
    "/kaggle/input/datasets/mavislei/"
    "nuscenes-dark-108/nuscenes_night_dark_subset_108/gt"
)

# Supplementary random nighttime sample
COLOR_DIR_FULL = Path(
    "/kaggle/input/datasets/mavislei/nuscenes-set/color"
)
GT_DIR_FULL = Path(
    "/kaggle/input/datasets/mavislei/nuscenes-set/gt"
)


for required_dir in [
    COLOR_DIR_DARK,
    GT_DIR_DARK,
    COLOR_DIR_FULL,
    GT_DIR_FULL
]:
    if not required_dir.is_dir():
        raise FileNotFoundError(f"Dataset directory not found: {required_dir}")


sampled_dark = sorted(
    f.name for f in COLOR_DIR_DARK.iterdir()
    if f.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

random.seed(42)
all_full_images = sorted(
    f.name for f in COLOR_DIR_FULL.iterdir()
    if f.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
sampled_full = random.sample(all_full_images, 200)


print(f"Primary low-brightness subset: {len(sampled_dark)} images")
print(f"Supplementary random sample: {len(sampled_full)} images")


Primary low-brightness subset: 108 images
Supplementary random sample: 200 images


## 4. Load Models and Define Processing Functions


In [30]:
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_ID = "depth-anything/Depth-Anything-V2-Metric-Outdoor-Small-hf"

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
metric_model = AutoModelForDepthEstimation.from_pretrained(MODEL_ID)
metric_model = metric_model.to(DEVICE).eval()

print(f"Metric depth model loaded on {DEVICE}")


def run_metric_depth(image_bgr):
    """Run metric depth inference and return a depth map in metres."""
    if image_bgr is None:
        raise ValueError("Input image is None.")

    image_rgb = Image.fromarray(
        cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    )

    inputs = processor(
        images=image_rgb,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        outputs = metric_model(**inputs)
        depth = outputs.predicted_depth

    depth = F.interpolate(
        depth.unsqueeze(1),
        size=image_rgb.size[::-1],
        mode="bicubic",
        align_corners=False
    ).squeeze().cpu().numpy().astype(np.float32)

    return depth


def depth_to_vis(depth):
    """Convert raw metric depth to a colour visualisation."""
    depth = np.asarray(depth, dtype=np.float32)
    valid = np.isfinite(depth)

    if valid.sum() == 0:
        return np.zeros((*depth.shape, 3), dtype=np.uint8)

    dmin = depth[valid].min()
    dmax = depth[valid].max()
    norm = np.zeros_like(depth, dtype=np.float32)
    norm[valid] = (depth[valid] - dmin) / (dmax - dmin + 1e-8)
    depth_u8 = np.clip(norm * 255.0, 0, 255).astype(np.uint8)
    return cv2.applyColorMap(depth_u8, cv2.COLORMAP_TURBO)


def apply_gamma(image_bgr, gamma=2.2):
    inv_gamma = 1.0 / gamma
    table = np.array([
        ((i / 255.0) ** inv_gamma) * 255
        for i in range(256)
    ]).astype(np.uint8)
    return cv2.LUT(image_bgr, table)


def apply_clahe(image_bgr):
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )
    l_enhanced = clahe.apply(l_channel)

    enhanced_lab = cv2.merge([
        l_enhanced,
        a_channel,
        b_channel
    ])
    return cv2.cvtColor(
        enhanced_lab,
        cv2.COLOR_LAB2BGR
    )


def apply_msr(
    image_bgr,
    sigmas=(15, 80, 250)
):
    """
    Multi-Scale Retinex with per-channel Min-Max normalization.
    """
    image = image_bgr.astype(np.float32) + 1.0
    msr = np.zeros_like(image, dtype=np.float32)

    for sigma in sigmas:
        blurred = cv2.GaussianBlur(
            image,
            (0, 0),
            sigma
        )

        msr += (
            np.log(image)
            - np.log(blurred + 1e-6)
        )

    msr /= len(sigmas)

    output = np.zeros_like(
        msr,
        dtype=np.float32
    )

    for channel_idx in range(3):
        channel = msr[:, :, channel_idx]

        channel_min = channel.min()
        channel_max = channel.max()

        if channel_max - channel_min < 1e-8:
            output[:, :, channel_idx] = 0
            continue

        output[:, :, channel_idx] = (
            (channel - channel_min)
            / (channel_max - channel_min)
            * 255.0
        )

    return np.clip(
        output,
        0,
        255
    ).astype(np.uint8)


def load_llformer():
    """Load the official LLFormer architecture and LOL checkpoint."""
    from model.LLFormer import LLFormer as LLFormerModel

    model = LLFormerModel(
        inp_channels=3,
        out_channels=3,
        dim=16,
        num_blocks=[2, 4, 8, 16],
        num_refinement_blocks=2,
        heads=[1, 2, 4, 8],
        ffn_expansion_factor=2.66,
        bias=False,
        LayerNorm_type="WithBias",
        attention=True,
        skip=False
    )

    checkpoint = torch.load(
        str(weights_path),
        map_location="cpu"
    )
    state_dict = checkpoint["state_dict"]

    try:
        model.load_state_dict(state_dict)
    except RuntimeError:
        cleaned_state_dict = OrderedDict()
        for key, value in state_dict.items():
            cleaned_key = (
                key[7:] if key.startswith("module.") else key
            )
            cleaned_state_dict[cleaned_key] = value

        model.load_state_dict(cleaned_state_dict)

    return model.to(DEVICE).eval()


def apply_llformer(image_bgr, llformer_model, max_size=512):
    """Enhance one image using LLFormer."""
    if image_bgr is None:
        raise ValueError("Input image is None.")

    original_height, original_width = image_bgr.shape[:2]
    resized = image_bgr

    height, width = resized.shape[:2]
    if max(height, width) > max_size:
        scale = max_size / max(height, width)
        resized = cv2.resize(
            resized,
            (
                max(1, int(round(width * scale))),
                max(1, int(round(height * scale)))
            ),
            interpolation=cv2.INTER_AREA
        )

    image_rgb = cv2.cvtColor(
        resized,
        cv2.COLOR_BGR2RGB
    )

    input_tensor = TF.to_tensor(
        Image.fromarray(image_rgb)
    ).unsqueeze(0).to(DEVICE)

    height, width = input_tensor.shape[2:]
    multiple = 16

    padded_height = (
        (height + multiple - 1) // multiple
    ) * multiple
    padded_width = (
        (width + multiple - 1) // multiple
    ) * multiple

    input_tensor = F.pad(
        input_tensor,
        (
            0,
            padded_width - width,
            0,
            padded_height - height
        ),
        mode="reflect"
    )

    with torch.no_grad():
        output = llformer_model(input_tensor)

    output = torch.clamp(
        output,
        0,
        1
    )[:, :, :height, :width]

    output = img_as_ubyte(
        output.permute(
            0, 2, 3, 1
        ).cpu().numpy()[0]
    )

    output = cv2.cvtColor(
        output,
        cv2.COLOR_RGB2BGR
    )

    output = cv2.resize(
        output,
        (original_width, original_height),
        interpolation=cv2.INTER_LINEAR
    )

    return output


def compute_metrics(prediction, reference_depth):
    """
    Compute MAE, RMSE, and AbsRel using the original evaluation mask:
    valid reference depth greater than 0 m and below 60 m.
    """
    prediction = np.asarray(
        prediction,
        dtype=np.float32
    )
    reference_depth = np.asarray(
        reference_depth,
        dtype=np.float32
    )

    if prediction.shape != reference_depth.shape:
        prediction = cv2.resize(
            prediction,
            (
                reference_depth.shape[1],
                reference_depth.shape[0]
            ),
            interpolation=cv2.INTER_CUBIC
        )

    mask = (
        (reference_depth > 0)
        & (reference_depth < 60)
        & np.isfinite(reference_depth)
        & np.isfinite(prediction)
        & (prediction > 0)
    )

    valid_pixels = int(mask.sum())
    if valid_pixels == 0:
        return None

    pred_valid = prediction[mask]
    gt_valid = reference_depth[mask]

    mae = float(
        np.mean(np.abs(pred_valid - gt_valid))
    )
    rmse = float(
        np.sqrt(
            np.mean(
                (pred_valid - gt_valid) ** 2
            )
        )
    )
    absrel = float(
        np.mean(
            np.abs(pred_valid - gt_valid)
            / gt_valid
        )
    )

    return {
        "mae": mae,
        "rmse": rmse,
        "absrel": absrel,
        "valid_pixels": valid_pixels
    }


PIPELINES = [
    "dark",
    "gamma",
    "clahe",
    "msr",
    "llformer"
]

print("All processing functions are ready.")


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

Metric depth model loaded on cuda
All processing functions are ready.


## 5. Pre-compute LLFormer Images

LLFormer is loaded once, used for both datasets, then unloaded before metric-depth inference to reduce GPU memory use.


In [31]:
LLFORMER_DARK_DIR = WORK_DIR / "llformer_enhanced_dark108"
LLFORMER_FULL_DIR = WORK_DIR / "llformer_enhanced_full200"

LLFORMER_DARK_DIR.mkdir(parents=True, exist_ok=True)
LLFORMER_FULL_DIR.mkdir(parents=True, exist_ok=True)


def precompute_llformer_images(
    image_names,
    input_dir,
    output_dir,
    description
):
    model = load_llformer()

    try:
        for filename in tqdm(
            image_names,
            desc=description
        ):
            output_path = output_dir / filename

            if output_path.is_file():
                continue

            image = cv2.imread(
                str(input_dir / filename)
            )

            if image is None:
                raise FileNotFoundError(
                    f"Unable to read image: "
                    f"{input_dir / filename}"
                )

            enhanced = apply_llformer(
                image,
                model
            )

            success = cv2.imwrite(
                str(output_path),
                enhanced
            )

            if not success:
                raise IOError(
                    f"Unable to save image: {output_path}"
                )
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


precompute_llformer_images(
    sampled_dark,
    COLOR_DIR_DARK,
    LLFORMER_DARK_DIR,
    "LLFormer: primary dark subset"
)

precompute_llformer_images(
    sampled_full,
    COLOR_DIR_FULL,
    LLFORMER_FULL_DIR,
    "LLFormer: supplementary full sample"
)

print("LLFormer pre-computation completed.")


LLFormer: supplementary full sample: 100%|██████████| 200/200 [00:00<00:00, 62073.46it/s]

LLFormer pre-computation completed.


## 6. Primary Evaluation: 108-Image Low-Brightness Subset


In [32]:
OUTPUT_DIR_DARK = (
    WORK_DIR / "nuscenes_metric_results_dark108"
)
OUTPUT_DIR_DARK.mkdir(parents=True, exist_ok=True)


def evaluate_dataset(
    image_names,
    color_dir,
    gt_dir,
    llformer_dir,
    output_csv,
    description,
    save_outputs=True,
    output_root=None
):
    rows = []

    if save_outputs:
        if output_root is None:
            output_root = Path(output_csv).parent / "output"
        output_root = Path(output_root)
        output_root.mkdir(parents=True, exist_ok=True)

    for filename in tqdm(image_names, desc=description):
        image_path = color_dir / filename
        image_bgr = cv2.imread(str(image_path))
        if image_bgr is None:
            raise FileNotFoundError(f"Unable to read image: {image_path}")

        gt_path = gt_dir / f"{Path(filename).stem}.npy"
        if not gt_path.is_file():
            raise FileNotFoundError(f"Reference depth not found: {gt_path}")
        reference_depth = np.load(gt_path).astype(np.float32)

        llformer_path = llformer_dir / filename
        llformer_image = cv2.imread(str(llformer_path))
        if llformer_image is None:
            raise FileNotFoundError(f"LLFormer image not found: {llformer_path}")

        pipeline_images = {
            "dark": image_bgr,
            "gamma": apply_gamma(image_bgr),
            "clahe": apply_clahe(image_bgr),
            "msr": apply_msr(image_bgr),
            "llformer": llformer_image
        }

        row = {"filename": filename}

        if save_outputs:
            image_output_dir = output_root / Path(filename).stem
            rgb_dir = image_output_dir / "rgb"
            depth_dir = image_output_dir / "depth"
            rgb_dir.mkdir(parents=True, exist_ok=True)
            depth_dir.mkdir(parents=True, exist_ok=True)

        for pipeline_name, pipeline_image in pipeline_images.items():
            prediction = run_metric_depth(pipeline_image)
            metrics = compute_metrics(prediction, reference_depth)

            if metrics is None:
                row[f"{pipeline_name}_mae"] = np.nan
                row[f"{pipeline_name}_rmse"] = np.nan
                row[f"{pipeline_name}_absrel"] = np.nan
                row[f"{pipeline_name}_valid_pixels"] = 0
            else:
                for metric_name, metric_value in metrics.items():
                    row[f"{pipeline_name}_{metric_name}"] = metric_value

            if save_outputs:
                rgb_path = rgb_dir / f"{pipeline_name}.png"
                depth_npy_path = depth_dir / f"{pipeline_name}.npy"
                depth_png_path = depth_dir / f"{pipeline_name}.png"

                if not cv2.imwrite(str(rgb_path), pipeline_image):
                    raise IOError(f"Unable to save RGB image: {rgb_path}")
                np.save(depth_npy_path, prediction)
                depth_vis = depth_to_vis(prediction)
                if not cv2.imwrite(str(depth_png_path), depth_vis):
                    raise IOError(f"Unable to save depth image: {depth_png_path}")

        rows.append(row)

    dataframe = pd.DataFrame(rows)
    dataframe.to_csv(output_csv, index=False)
    return dataframe


def print_mean_results(dataframe, title):
    print(f"\n{title}")
    print("-" * len(title))
    for pipeline_name in PIPELINES:
        mae = dataframe[f"{pipeline_name}_mae"].mean()
        rmse = dataframe[f"{pipeline_name}_rmse"].mean()
        absrel = dataframe[f"{pipeline_name}_absrel"].mean()
        print(
            f"{pipeline_name:10s}: "
            f"MAE={mae:.4f}, "
            f"RMSE={rmse:.4f}, "
            f"AbsRel={absrel:.4f}"
        )


df_dark = evaluate_dataset(
    image_names=sampled_dark,
    color_dir=COLOR_DIR_DARK,
    gt_dir=GT_DIR_DARK,
    llformer_dir=LLFORMER_DARK_DIR,
    output_csv=OUTPUT_DIR_DARK / "metrics_dark108_per_image.csv",
    description="Evaluating primary dark subset",
    save_outputs=True,
    output_root=OUTPUT_DIR_DARK / "output"
)

print_mean_results(df_dark, "Primary 108-image low-brightness subset")


Evaluating primary dark subset: 100%|██████████| 108/108 [01:57<00:00,  1.09s/it]


Primary 108-image low-brightness subset
---------------------------------------
dark      : MAE=9.6603, RMSE=14.8240, AbsRel=0.3937
gamma     : MAE=11.3491, RMSE=16.7629, AbsRel=0.4641
clahe     : MAE=9.9502, RMSE=15.3179, AbsRel=0.3930
msr       : MAE=11.8616, RMSE=17.3755, AbsRel=0.4811
llformer  : MAE=11.1183, RMSE=16.5742, AbsRel=0.4474


## 7. Exploratory Blending on the 108-Image Subset

Fixed-alpha blending uses:

$$
I_{\mathrm{blend}}
=
\alpha I_{\mathrm{LLFormer}}
+
(1-\alpha) I_{\mathrm{original}}
$$

The spatially adaptive version assigns a lower LLFormer weight near strong image edges.


In [33]:
FIXED_ALPHAS = [0.2, 0.4, 0.6, 0.8]


def compute_alpha_map(
    image_bgr,
    alpha_min=0.05,
    alpha_max=0.30
):
    gray = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2GRAY
    ).astype(np.float32)

    gradient_x = cv2.Sobel(
        gray,
        cv2.CV_32F,
        1,
        0,
        ksize=3
    )
    gradient_y = cv2.Sobel(
        gray,
        cv2.CV_32F,
        0,
        1,
        ksize=3
    )

    edge_magnitude = np.sqrt(
        gradient_x ** 2
        + gradient_y ** 2
    )

    edge_magnitude = cv2.GaussianBlur(
        edge_magnitude,
        (5, 5),
        0
    )

    edge_normalised = (
        edge_magnitude
        - edge_magnitude.min()
    ) / (
        edge_magnitude.max()
        - edge_magnitude.min()
        + 1e-8
    )

    alpha_map = (
        alpha_min
        + (alpha_max - alpha_min)
        * (1.0 - edge_normalised)
    )

    return alpha_map[..., None].astype(
        np.float32
    )


# Separate directory for blending outputs
BLEND_OUTPUT_DIR = (
    OUTPUT_DIR_DARK
    / "output_blending"
)
BLEND_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


fixed_rows = []
adaptive_rows = []

for filename in tqdm(
    sampled_dark,
    desc="Blending evaluation: dark subset"
):
    image_bgr = cv2.imread(
        str(COLOR_DIR_DARK / filename)
    )
    llformer_image = cv2.imread(
        str(LLFORMER_DARK_DIR / filename)
    )

    if image_bgr is None:
        raise FileNotFoundError(
            f"Unable to read original image: "
            f"{COLOR_DIR_DARK / filename}"
        )

    if llformer_image is None:
        raise FileNotFoundError(
            f"Unable to read LLFormer image: "
            f"{LLFORMER_DARK_DIR / filename}"
        )

    gt_path = (
        GT_DIR_DARK
        / f"{Path(filename).stem}.npy"
    )

    if not gt_path.is_file():
        raise FileNotFoundError(
            f"Unable to find reference depth: "
            f"{gt_path}"
        )

    reference_depth = np.load(
        gt_path
    ).astype(np.float32)

    # Create one output folder per image
    image_output_dir = (
        BLEND_OUTPUT_DIR
        / Path(filename).stem
    )
    rgb_output_dir = (
        image_output_dir
        / "rgb"
    )
    depth_output_dir = (
        image_output_dir
        / "depth"
    )

    rgb_output_dir.mkdir(
        parents=True,
        exist_ok=True
    )
    depth_output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    fixed_row = {
        "filename": filename
    }

    # ---------------------------------
    # Fixed-alpha blending
    # ---------------------------------
    for alpha in FIXED_ALPHAS:
        blended = cv2.addWeighted(
            llformer_image,
            alpha,
            image_bgr,
            1.0 - alpha,
            0
        )

        prediction = run_metric_depth(
            blended
        )

        metrics = compute_metrics(
            prediction,
            reference_depth
        )

        for metric_name in [
            "mae",
            "rmse",
            "absrel",
            "valid_pixels"
        ]:
            value = (
                metrics[metric_name]
                if metrics is not None
                else np.nan
            )

            fixed_row[
                f"blend_{alpha}_{metric_name}"
            ] = value

        method_name = (
            f"fixed_{alpha:.1f}"
        )

        rgb_path = (
            rgb_output_dir
            / f"{method_name}.png"
        )
        depth_npy_path = (
            depth_output_dir
            / f"{method_name}.npy"
        )
        depth_png_path = (
            depth_output_dir
            / f"{method_name}.png"
        )

        if not cv2.imwrite(
            str(rgb_path),
            blended
        ):
            raise IOError(
                f"Unable to save blending image: "
                f"{rgb_path}"
            )

        np.save(
            depth_npy_path,
            prediction
        )

        depth_visualisation = depth_to_vis(
            prediction
        )

        if not cv2.imwrite(
            str(depth_png_path),
            depth_visualisation
        ):
            raise IOError(
                f"Unable to save depth image: "
                f"{depth_png_path}"
            )

    fixed_rows.append(
        fixed_row
    )

    # ---------------------------------
    # Adaptive blending
    # ---------------------------------
    alpha_map = compute_alpha_map(
        image_bgr,
        alpha_min=0.05,
        alpha_max=0.30
    )

    adaptive_blended = (
        alpha_map
        * llformer_image.astype(np.float32)
        + (1.0 - alpha_map)
        * image_bgr.astype(np.float32)
    )

    adaptive_blended = np.clip(
        adaptive_blended,
        0,
        255
    ).astype(np.uint8)

    adaptive_prediction = run_metric_depth(
        adaptive_blended
    )

    adaptive_metrics = compute_metrics(
        adaptive_prediction,
        reference_depth
    )

    adaptive_row = {
        "filename": filename
    }

    for metric_name in [
        "mae",
        "rmse",
        "absrel",
        "valid_pixels"
    ]:
        adaptive_row[
            f"adaptive_{metric_name}"
        ] = (
            adaptive_metrics[metric_name]
            if adaptive_metrics is not None
            else np.nan
        )

    adaptive_rows.append(
        adaptive_row
    )

    adaptive_rgb_path = (
        rgb_output_dir
        / "adaptive.png"
    )
    adaptive_depth_npy_path = (
        depth_output_dir
        / "adaptive.npy"
    )
    adaptive_depth_png_path = (
        depth_output_dir
        / "adaptive.png"
    )

    if not cv2.imwrite(
        str(adaptive_rgb_path),
        adaptive_blended
    ):
        raise IOError(
            f"Unable to save adaptive image: "
            f"{adaptive_rgb_path}"
        )

    np.save(
        adaptive_depth_npy_path,
        adaptive_prediction
    )

    adaptive_depth_visualisation = (
        depth_to_vis(
            adaptive_prediction
        )
    )

    if not cv2.imwrite(
        str(adaptive_depth_png_path),
        adaptive_depth_visualisation
    ):
        raise IOError(
            f"Unable to save adaptive depth image: "
            f"{adaptive_depth_png_path}"
        )


df_fixed_dark = pd.DataFrame(
    fixed_rows
)
df_adaptive_dark = pd.DataFrame(
    adaptive_rows
)

fixed_csv_path = (
    OUTPUT_DIR_DARK
    / "blend_fixed_alpha_dark108.csv"
)
adaptive_csv_path = (
    OUTPUT_DIR_DARK
    / "blend_adaptive_dark108.csv"
)

df_fixed_dark.to_csv(
    fixed_csv_path,
    index=False
)
df_adaptive_dark.to_csv(
    adaptive_csv_path,
    index=False
)


print("\nFixed-alpha blending results")
print("----------------------------")

for alpha in FIXED_ALPHAS:
    print(
        f"alpha={alpha:.1f}: "
        f"MAE="
        f"{df_fixed_dark[f'blend_{alpha}_mae'].mean():.4f}, "
        f"RMSE="
        f"{df_fixed_dark[f'blend_{alpha}_rmse'].mean():.4f}, "
        f"AbsRel="
        f"{df_fixed_dark[f'blend_{alpha}_absrel'].mean():.4f}"
    )


print("\nAdaptive blending result")
print("------------------------")
print(
    f"MAE="
    f"{df_adaptive_dark['adaptive_mae'].mean():.4f}, "
    f"RMSE="
    f"{df_adaptive_dark['adaptive_rmse'].mean():.4f}, "
    f"AbsRel="
    f"{df_adaptive_dark['adaptive_absrel'].mean():.4f}"
)


print("\nSaved files")
print("-----------")
print(
    f"Fixed-alpha CSV: "
    f"{fixed_csv_path}"
)
print(
    f"Adaptive CSV: "
    f"{adaptive_csv_path}"
)
print(
    f"Blending images and depths: "
    f"{BLEND_OUTPUT_DIR}"
)

Blending evaluation: dark subset: 100%|██████████| 108/108 [00:40<00:00,  2.67it/s]


Fixed-alpha blending results
----------------------------
alpha=0.2: MAE=9.7528, RMSE=14.8859, AbsRel=0.4002
alpha=0.4: MAE=10.0474, RMSE=15.2366, AbsRel=0.4116
alpha=0.6: MAE=10.4093, RMSE=15.6854, AbsRel=0.4245
alpha=0.8: MAE=10.7780, RMSE=16.1487, AbsRel=0.4365

Adaptive blending result
------------------------
MAE=9.9066, RMSE=15.0567, AbsRel=0.4071

Saved files
-----------
Fixed-alpha CSV: /kaggle/working/nuscenes_metric_results_dark108/blend_fixed_alpha_dark108.csv
Adaptive CSV: /kaggle/working/nuscenes_metric_results_dark108/blend_adaptive_dark108.csv
Blending images and depths: /kaggle/working/nuscenes_metric_results_dark108/output_blending


## 8. Supplementary Evaluation: Random 200-Image Nighttime Sample

This section checks whether the main conclusion also holds across a broader nighttime sample.


In [34]:
OUTPUT_DIR_FULL = (
    WORK_DIR / "nuscenes_metric_results_full200"
)
OUTPUT_DIR_FULL.mkdir(
    parents=True,
    exist_ok=True
)

df_full = evaluate_dataset(
    image_names=sampled_full,
    color_dir=COLOR_DIR_FULL,
    gt_dir=GT_DIR_FULL,
    llformer_dir=LLFORMER_FULL_DIR,
    output_csv=(
        OUTPUT_DIR_FULL
        / "metrics_full200_per_image.csv"
    ),
    description="Evaluating supplementary full sample",
    save_outputs=True,
    output_root=OUTPUT_DIR_FULL / "output"
)

print_mean_results(
    df_full,
    "Supplementary random 200-image sample"
)


Evaluating supplementary full sample: 100%|██████████| 200/200 [03:36<00:00,  1.08s/it]


Supplementary random 200-image sample
-------------------------------------
dark      : MAE=8.6483, RMSE=12.1887, AbsRel=0.4608
gamma     : MAE=9.2643, RMSE=13.1589, AbsRel=0.4714
clahe     : MAE=8.6655, RMSE=12.4427, AbsRel=0.4403
msr       : MAE=9.3663, RMSE=13.0942, AbsRel=0.4848
llformer  : MAE=9.5870, RMSE=13.8049, AbsRel=0.4594


## 9. Save Summary Tables and Package Results


In [35]:
def build_summary_table(dataframe):
    rows = []

    for pipeline_name in PIPELINES:
        rows.append({
            "pipeline": pipeline_name,
            "mean_mae": dataframe[
                f"{pipeline_name}_mae"
            ].mean(),
            "mean_rmse": dataframe[
                f"{pipeline_name}_rmse"
            ].mean(),
            "mean_absrel": dataframe[
                f"{pipeline_name}_absrel"
            ].mean(),
            "n_images": len(dataframe)
        })

    return pd.DataFrame(rows)


summary_dark = build_summary_table(
    df_dark
)
summary_full = build_summary_table(
    df_full
)

summary_dark.to_csv(
    OUTPUT_DIR_DARK
    / "summary_dark108.csv",
    index=False
)

summary_full.to_csv(
    OUTPUT_DIR_FULL
    / "summary_full200.csv",
    index=False
)


archive_dark = shutil.make_archive(
    str(
        WORK_DIR
        / "nuscenes_metric_results_dark108"
    ),
    "zip",
    str(OUTPUT_DIR_DARK)
)

archive_full = shutil.make_archive(
    str(
        WORK_DIR
        / "nuscenes_metric_results_full200"
    ),
    "zip",
    str(OUTPUT_DIR_FULL)
)


print(f"Primary results archive: {archive_dark}")
print(f"Supplementary results archive: {archive_full}")


Primary results archive: /kaggle/working/nuscenes_metric_results_dark108.zip
Supplementary results archive: /kaggle/working/nuscenes_metric_results_full200.zip
